# 03 — Los 3 SKILLs de Tesorería

**Autor:** Gabriel Untiveros | **Fecha:** 2026-05-26

---

## Qué construimos aquí

En el NB 02 creamos las **herramientas** (scripts Python).  
En este NB creamos las **reglas de uso** de esas herramientas: los SKILLs `.md`.

Un SKILL no es documentación — es una instrucción operativa que el agente lee bajo demanda.  
No está siempre en el prompt (desperdicia tokens). Se carga solo cuando la tarea lo requiere.

| SKILL | Input | Output | Adaptado de |
|---|---|---|---|
| `forecast-cashflow/SKILL.md` | cuenta_id, horizon | forecast_saldo, dias_de_caja | `forecasting/SKILL.md` |
| `alerta-tesoreria/SKILL.md` | dias_de_caja, saldo_actual, saldo_minimo | nivel_alerta, accion | `reorder-policy/SKILL.md` |
| `reporte-semanal/SKILL.md` | outputs de los 2 anteriores | markdown del reporte | `weekly-report/SKILL.md` |

## La cadena de dependencias

```
batch_dias_de_caja.py
   └─ produce: {dias_de_caja, saldo_actual, confidence, flags} por cuenta
        └─ alerta-tesoreria/SKILL.md lo consume
             └─ produce: {nivel_alerta, accion_recomendada} por cuenta
                  └─ reporte-semanal/SKILL.md lo consume
                       └─ produce: markdown del reporte ejecutivo
                            └─ UN script Python (generar_reporte.py) hace todo esto
```

In [1]:
import json, subprocess, sys
import pandas as pd
from pathlib import Path
from datetime import date

BASE        = Path('..').resolve()
DATA_PATH   = BASE / 'data' / 'movimientos_diarios.csv'
SKILLS_BASE = BASE / '.claude' / 'skills'

# Verificar que el NB 02 se ejecutó
script_forecast = SKILLS_BASE / 'forecast-cashflow' / 'rolling_mean_cashflow.py'
assert script_forecast.exists(), 'Falta rolling_mean_cashflow.py. Ejecuta primero NB 02.'
assert (SKILLS_BASE / 'forecast-cashflow' / 'SKILL.md').exists(), 'Falta SKILL.md del forecast.'

print('Prerequisitos OK')
print(f'  Skills existentes: {[p.parent.name for p in SKILLS_BASE.glob("*/SKILL.md")]}')

Prerequisitos OK
  Skills existentes: ['forecast-cashflow']


---
## SKILL 1 — `forecast-cashflow/SKILL.md` (ya existe desde NB 02)

Verificamos que el contrato de output es correcto antes de usarlo como input del SKILL de alertas.

In [2]:
# Ejecutar batch y capturar el output — este es el INPUT del skill de alertas
batch_script = SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py'
result = subprocess.run(
    [sys.executable, str(batch_script)],
    capture_output=True, text=True
)
assert result.returncode == 0, f'Error en batch: {result.stderr}'

forecast_data = json.loads(result.stdout)

print('Output del batch forecast (input para alerta-tesoreria):')
print(json.dumps(forecast_data, indent=2, ensure_ascii=False))

Output del batch forecast (input para alerta-tesoreria):
[
  {
    "cuenta_id": "CUENTA-001",
    "moneda": "PEN",
    "saldo_actual": 356608.41,
    "saldo_minimo": 50000,
    "dias_de_caja": 999,
    "flujo_neto_dia": 1968.91,
    "bajo_minimo": false,
    "requiere_atencion": false,
    "confidence": 0.85
  },
  {
    "cuenta_id": "CUENTA-002",
    "moneda": "PEN",
    "saldo_actual": 224330.79,
    "saldo_minimo": 10000,
    "dias_de_caja": 999,
    "flujo_neto_dia": 1289.15,
    "bajo_minimo": false,
    "requiere_atencion": false,
    "confidence": 0.85
  },
  {
    "cuenta_id": "CUENTA-003",
    "moneda": "USD",
    "saldo_actual": 118639.42,
    "saldo_minimo": 5000,
    "dias_de_caja": 999,
    "flujo_neto_dia": 187.47,
    "bajo_minimo": false,
    "requiere_atencion": false,
    "confidence": 0.85
  }
]


---
## SKILL 2 — `alerta-tesoreria/SKILL.md`

Adaptado de: `cwc-workshops/agent-decomposition/.claude/skills/reorder-policy/SKILL.md`

| Workshop (inventario) | Tesorería |
|---|---|
| `on_hand < reorder_point` → reordenar | `saldo_actual < saldo_minimo` → alerta CRÍTICO |
| `days_of_cover < lead_time` → expeditar | `dias_de_caja < 7` → alerta ALTO |
| `confidence < 0.6` → escalar | `confidence < 0.6` → escalar (mismo) |
| Output: `{reorder, qty, supplier_id}` | Output: `{nivel_alerta, accion_recomendada}` |

In [3]:
SKILL_ALERTA = '''
---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere accion urgente.
  Cargar cuando la tarea sea: revisar saldos, alertas, hay liquidez,
  cual cuenta esta critica, o despues de ejecutar el forecast.
---

# Politica de Alertas de Tesoreria

## Inputs necesarios

Vienen del output de `batch_dias_de_caja.py` o `rolling_mean_cashflow.py`:

```json
{
  "cuenta_id":     "CUENTA-001",
  "saldo_actual":   356608.41,
  "saldo_minimo":   50000,
  "dias_de_caja":   45.2,
  "flujo_neto_dia": 1968.91,
  "confidence":     0.85,
  "flags":          []
}
```

## Niveles de alerta (evaluar en este orden)

| Nivel | Condicion | Accion inmediata |
|---|---|---|
| **CRITICO** | `saldo_actual < saldo_minimo` | Notificar gerencia. Bloquear pagos no criticos. Activar linea de credito. |
| **ALTO** | `dias_de_caja < 7` | Transferencia urgente entre cuentas. Revisar pagos de la semana. |
| **MEDIO** | `dias_de_caja < 14` | Revisar egresos programados. Confirmar fondeos pendientes. |
| **OK** | `dias_de_caja >= 14` Y `saldo_actual >= saldo_minimo` | Sin accion requerida. |

## Formula de dias_de_caja

```
Si flujo_neto_dia < 0:
    dias_de_caja = saldo_actual / abs(flujo_neto_dia)
Si flujo_neto_dia >= 0:
    dias_de_caja = 999  (flujo positivo, sin limite en ventana actual)
```

**Importante:** `dias_de_caja = 999` no significa que la cuenta este perfecta,
significa que en los ultimos 14 dias los ingresos superaron a los egresos.
Puede haber pagos programados grandes que cambien esto — consultar el calendario de pagos.

## Regla de confidence

Si `confidence < 0.60` (historial insuficiente o datos anomalos):
- **No recomendar accion automatica** sobre esa cuenta.
- Incluir en el reporte con nivel `REVISAR` y los flags como contexto.
- El tesorero debe revisar manualmente.

## Output esperado por cuenta

```json
{
  "cuenta_id":          "CUENTA-001",
  "nivel_alerta":        "OK",
  "accion_recomendada": "Sin accion requerida",
  "prioridad":           4
}
```

Prioridades para ordenar el reporte: CRITICO=1, ALTO=2, MEDIO=3, OK=4, REVISAR=0.

## Cuando NO usar este skill

- Para medidas DAX de Power BI (usar dax-tesoreria)
- Para analisis de cuentas por cobrar o pagar (son dominios distintos)
- Para decisiones de inversion del excedente (requiere criterio del CFO)
'''

alerta_path = SKILLS_BASE / 'alerta-tesoreria'
alerta_path.mkdir(parents=True, exist_ok=True)
(alerta_path / 'SKILL.md').write_text(SKILL_ALERTA.strip(), encoding='utf-8')
print(f'SKILL creado: {alerta_path / "SKILL.md"}')

SKILL creado: D:\Proyecto_Gabriel\Skill_financiero\.claude\skills\alerta-tesoreria\SKILL.md


---
## Probar la lógica de alertas con los datos reales del forecast

Implementamos la función de alertas en Python para verificar que el SKILL.md
produce el output correcto con nuestros datos simulados.

In [4]:
def evaluar_alerta(row: dict) -> dict:
    """Aplica las reglas del SKILL alerta-tesoreria a un row del batch forecast."""
    cuenta_id  = row['cuenta_id']
    saldo      = row['saldo_actual']
    saldo_min  = row['saldo_minimo']
    dias       = row['dias_de_caja']
    confidence = row['confidence']

    if confidence < 0.60:
        return {'cuenta_id': cuenta_id, 'nivel_alerta': 'REVISAR',
                'accion_recomendada': 'Historial insuficiente — revision manual requerida',
                'prioridad': 0}
    if saldo < saldo_min:
        return {'cuenta_id': cuenta_id, 'nivel_alerta': 'CRITICO',
                'accion_recomendada': 'Notificar gerencia. Activar linea de credito.',
                'prioridad': 1}
    if dias < 7:
        return {'cuenta_id': cuenta_id, 'nivel_alerta': 'ALTO',
                'accion_recomendada': 'Transferencia urgente. Revisar pagos semana.',
                'prioridad': 2}
    if dias < 14:
        return {'cuenta_id': cuenta_id, 'nivel_alerta': 'MEDIO',
                'accion_recomendada': 'Revisar egresos programados. Confirmar fondeos.',
                'prioridad': 3}
    return {'cuenta_id': cuenta_id, 'nivel_alerta': 'OK',
            'accion_recomendada': 'Sin accion requerida',
            'prioridad': 4}

# Aplicar a cada cuenta del forecast
alertas = [evaluar_alerta(row) for row in forecast_data]
alertas.sort(key=lambda x: x['prioridad'])

print('Evaluacion de alertas por cuenta:')
print()
for a in alertas:
    icono = {'CRITICO': 'CRITICO', 'ALTO': 'ALTO', 'MEDIO': 'MEDIO',
             'OK': 'OK     ', 'REVISAR': 'REVISAR'}[a['nivel_alerta']]
    # Buscar datos del forecast para mostrar contexto
    fila = next(r for r in forecast_data if r['cuenta_id'] == a['cuenta_id'])
    print(f"[{icono}] {a['cuenta_id']} ({fila['moneda']})")
    print(f"         Saldo: {fila['moneda']} {fila['saldo_actual']:>12,.2f} | Min: {fila['saldo_minimo']:>10,.0f} | Dias caja: {fila['dias_de_caja']}")
    print(f"         Accion: {a['accion_recomendada']}")
    print()

Evaluacion de alertas por cuenta:

[OK     ] CUENTA-001 (PEN)
         Saldo: PEN   356,608.41 | Min:     50,000 | Dias caja: 999
         Accion: Sin accion requerida

[OK     ] CUENTA-002 (PEN)
         Saldo: PEN   224,330.79 | Min:     10,000 | Dias caja: 999
         Accion: Sin accion requerida

[OK     ] CUENTA-003 (USD)
         Saldo: USD   118,639.42 | Min:      5,000 | Dias caja: 999
         Accion: Sin accion requerida



---
## SKILL 3 — `reporte-semanal/SKILL.md`

Adaptado de: `cwc-workshops/agent-decomposition/.claude/skills/weekly-report/SKILL.md`

La instruccion mas importante que viene del workshop:

> **Genera el reporte escribiendo UN script Python que lea los CSVs y emita markdown.**
> **No hagas una tool call por cuenta — ese es exactamente el anti-patron que este skill reemplaza.**

In [5]:
SKILL_REPORTE = '''
---
name: reporte-semanal-tesoreria
description: >
  Estructura del reporte semanal de flujo de caja y alertas.
  Cargar cuando pidan: reporte semanal, resumen de tesoreria, informe del lunes,
  como estamos en caja, posicion consolidada.
---

# Reporte Semanal de Tesoreria

## Regla principal (del workshop)

Genera el reporte ejecutando UN script Python. No hagas tool calls individuales por cuenta.
El CSV puede tener meses de historial — un script lo procesa todo en una sola llamada.

```bash
python .claude/skills/reporte-semanal/generar_reporte.py
```

Ese script ya integra el forecast + las alertas + la posicion consolidada.

## Cadencia

| Cadencia | Trigger | Contenido |
|---|---|---|
| **Lunes** | reporte semanal, informe del lunes | Posicion completa: alertas + forecast + pagos criticos semana |
| **Diario** | revision diaria, el sweep, como estamos | Solo cuentas con alerta CRITICO o ALTO |
| **Ad hoc** | cualquier otra pregunta | Scope a lo que pidieron |

Si no se especifica, inferir por contexto. Default: reporte semanal completo.

## Estructura del reporte semanal (markdown)

```
# Reporte de Tesoreria — Semana del {{fecha}}

## Resumen Ejecutivo
{{N}} cuentas revisadas | {{N_alertas}} requieren atencion | Posicion total: S/ XX,XXX

## Cuentas en Alerta
| Cuenta | Banco | Moneda | Saldo actual | Minimo | Dias de caja | Nivel | Accion |
(solo cuentas con nivel CRITICO, ALTO o MEDIO)

## Cuentas OK
| Cuenta | Banco | Moneda | Saldo actual | Dias de caja | Forecast 14d |

## Posicion Consolidada
| Moneda | Saldo total | Variacion semana |

## Pagos Criticos Esta Semana
(cuando haya datos de pagos programados — pendiente de implementar)

## Notas del Agente
Observaciones relevantes del periodo (tendencias, anomalias, recomendaciones).
```

## Cuando usar el script vs generar a mano

- **Siempre usa el script** para el reporte completo — tiene toda la logica integrada.
- **Genera a mano** solo si el usuario pide una cuenta especifica o un dato puntual.
  En ese caso, ejecuta solo `rolling_mean_cashflow.py` para esa cuenta.

## Importante: dias_de_caja = 999

No significa que la cuenta sea perfecta. Significa que en los ultimos 14 dias
los ingresos superaron a los egresos. Mencionarlo como 'flujo positivo esta semana'
y recomendar revisar el calendario de pagos proximos.
'''

reporte_path = SKILLS_BASE / 'reporte-semanal'
reporte_path.mkdir(parents=True, exist_ok=True)
(reporte_path / 'SKILL.md').write_text(SKILL_REPORTE.strip(), encoding='utf-8')
print(f'SKILL creado: {reporte_path / "SKILL.md"}')

SKILL creado: D:\Proyecto_Gabriel\Skill_financiero\.claude\skills\reporte-semanal\SKILL.md


---
## El script `generar_reporte.py` — la Capa 2 del agente

Este es el script que el SKILL de reporte-semanal referencia.  
Integra todo: forecast + alertas + posicion consolidada → un markdown completo.  
Una sola ejecucion, sin tool calls intermedias.

In [6]:
SCRIPT_REPORTE = '''
#!/usr/bin/env python3
"""
Genera el reporte semanal de tesoreria en markdown.
Integra: forecast de caja + alertas + posicion consolidada.

Uso:    python generar_reporte.py
Output: markdown del reporte (stdout) o archivo .md
"""
import csv, json, sys
from pathlib import Path
from collections import defaultdict
from datetime import date

DATA        = Path(__file__).parent.parent.parent.parent / "data" / "movimientos_diarios.csv"
FECHA_HOY   = date.today().strftime("%d-%b-%Y")

# Saldos minimos operativos (configurar por cliente)
SALDO_MINIMO = {"CUENTA-001": 50_000, "CUENTA-002": 10_000, "CUENTA-003": 5_000}
NOMBRES      = {"CUENTA-001": "BCP", "CUENTA-002": "BBVA", "CUENTA-003": "Interbank"}

# ── Cargar y agrupar historial ────────────────────────────────────────────
historial = defaultdict(list)
for r in csv.DictReader(open(DATA)):
    historial[r["cuenta_id"]].append({
        "saldo_cierre": float(r["saldo_cierre"]),
        "ingresos_dia": float(r["ingresos_dia"]),
        "egresos_dia":  float(r["egresos_dia"]),
        "moneda":        r["moneda"],
    })

# ── Calcular forecast y alerta por cuenta ────────────────────────────────
cuentas = []
for cuenta_id, registros in historial.items():
    saldos   = [r["saldo_cierre"] for r in registros]
    ingresos = [r["ingresos_dia"]  for r in registros]
    egresos  = [r["egresos_dia"]   for r in registros]
    moneda   = registros[0]["moneda"]

    flujos      = [i - e for i, e in zip(ingresos[-14:], egresos[-14:])]
    flujo_dia   = sum(flujos) / max(len(flujos), 1)
    saldo_act   = saldos[-1]
    saldo_ant   = saldos[-8] if len(saldos) >= 8 else saldos[0]  # hace 7 dias
    variacion   = saldo_act - saldo_ant
    forecast    = round(saldo_act + flujo_dia * 14, 2)
    dias_caja   = round(saldo_act / abs(flujo_dia), 1) if flujo_dia < 0 else 999
    confidence  = 0.85 if len(saldos) >= 14 else 0.60
    saldo_min   = SALDO_MINIMO.get(cuenta_id, 0)

    # Nivel de alerta
    if confidence < 0.60:
        nivel, prioridad = "REVISAR", 0
        accion = "Historial insuficiente"
    elif saldo_act < saldo_min:
        nivel, prioridad = "CRITICO", 1
        accion = "Notificar gerencia. Activar linea de credito."
    elif dias_caja < 7:
        nivel, prioridad = "ALTO", 2
        accion = "Transferencia urgente. Revisar pagos semana."
    elif dias_caja < 14:
        nivel, prioridad = "MEDIO", 3
        accion = "Revisar egresos programados."
    else:
        nivel, prioridad = "OK", 4
        accion = "Sin accion requerida"

    cuentas.append({
        "cuenta_id": cuenta_id, "banco": NOMBRES.get(cuenta_id, "-"),
        "moneda": moneda, "saldo_actual": saldo_act, "saldo_anterior": saldo_ant,
        "variacion": variacion, "saldo_minimo": saldo_min,
        "dias_de_caja": dias_caja, "flujo_dia": flujo_dia, "forecast_14d": forecast,
        "nivel_alerta": nivel, "accion": accion, "prioridad": prioridad,
    })

cuentas.sort(key=lambda x: x["prioridad"])

# ── Posicion consolidada por moneda ──────────────────────────────────────
pos_pen = sum(c["saldo_actual"] for c in cuentas if c["moneda"] == "PEN")
pos_usd = sum(c["saldo_actual"] for c in cuentas if c["moneda"] == "USD")

n_alertas = sum(1 for c in cuentas if c["prioridad"] <= 3)

# ── Generar markdown ─────────────────────────────────────────────────────
md = []
md.append(f"# Reporte de Tesoreria — Semana del {FECHA_HOY}")
md.append("")
md.append("## Resumen Ejecutivo")
md.append(f"- **Cuentas revisadas:** {len(cuentas)}")
md.append(f"- **Requieren atencion:** {n_alertas}")
md.append(f"- **Posicion PEN:** S/ {pos_pen:,.2f}")
md.append(f"- **Posicion USD:** USD {pos_usd:,.2f}")
md.append("")

# Cuentas en alerta
alertas = [c for c in cuentas if c["prioridad"] <= 3]
if alertas:
    md.append("## Cuentas en Alerta")
    md.append("| Cuenta | Banco | Moneda | Saldo actual | Minimo | Dias de caja | Nivel | Accion |")
    md.append("|---|---|---|---|---|---|---|---|")
    for c in alertas:
        dias_str = str(c["dias_de_caja"]) if c["dias_de_caja"] < 999 else "flujo+"
        md.append(f"| {c['cuenta_id']} | {c['banco']} | {c['moneda']} | "
                  f"{c['moneda']} {c['saldo_actual']:>12,.0f} | "
                  f"{c['saldo_minimo']:>10,.0f} | "
                  f"{dias_str} | **{c['nivel_alerta']}** | {c['accion']} |")
    md.append("")

# Cuentas OK
ok = [c for c in cuentas if c["prioridad"] > 3]
if ok:
    md.append("## Cuentas OK")
    md.append("| Cuenta | Banco | Moneda | Saldo actual | Variacion 7d | Forecast 14d |")
    md.append("|---|---|---|---|---|---|")
    for c in ok:
        var_str = f"+{c['variacion']:,.0f}" if c["variacion"] >= 0 else f"{c['variacion']:,.0f}"
        md.append(f"| {c['cuenta_id']} | {c['banco']} | {c['moneda']} | "
                  f"{c['moneda']} {c['saldo_actual']:>12,.0f} | "
                  f"{var_str} | {c['moneda']} {c['forecast_14d']:>12,.0f} |")
    md.append("")

# Posicion consolidada
md.append("## Posicion Consolidada")
md.append("| Moneda | Saldo total |")
md.append("|---|---|")
md.append(f"| PEN | S/ {pos_pen:,.2f} |")
md.append(f"| USD | USD {pos_usd:,.2f} |")
md.append("")

# Notas del agente
md.append("## Notas del Agente")
notas = []
for c in cuentas:
    if c["dias_de_caja"] == 999:
        notas.append(f"- {c['cuenta_id']}: flujo neto positivo esta semana ({c['moneda']} {c['flujo_dia']:+,.0f}/dia). Revisar pagos proximos.")
if not notas:
    notas = ["- Sin observaciones adicionales."]
md.extend(notas)

print(chr(10).join(md))
'''

script_path = reporte_path / 'generar_reporte.py'
script_path.write_text(SCRIPT_REPORTE.strip(), encoding='utf-8')
print(f'Script guardado: {script_path}')

Script guardado: D:\Proyecto_Gabriel\Skill_financiero\.claude\skills\reporte-semanal\generar_reporte.py


---
## Ejecutar y ver el reporte completo

In [7]:
result_reporte = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True
)

if result_reporte.returncode == 0:
    reporte_md = result_reporte.stdout
    print(reporte_md)
    # Guardar tambien en data/
    out_file = BASE / 'data' / 'reporte_semanal_ultimo.md'
    out_file.write_text(reporte_md, encoding='utf-8')
    print(f'Reporte guardado: {out_file}')
else:
    print('ERROR:', result_reporte.stderr)

# Reporte de Tesoreria â€” Semana del 26-May-2026

## Resumen Ejecutivo
- **Cuentas revisadas:** 3
- **Requieren atencion:** 0
- **Posicion PEN:** S/ 580,939.20
- **Posicion USD:** USD 118,639.42

## Cuentas OK
| Cuenta | Banco | Moneda | Saldo actual | Variacion 7d | Forecast 14d |
|---|---|---|---|---|---|
| CUENTA-001 | BCP | PEN | PEN      356,608 | +8,163 | PEN      384,173 |
| CUENTA-002 | BBVA | PEN | PEN      224,331 | +8,425 | PEN      242,379 |
| CUENTA-003 | Interbank | USD | USD      118,639 | -3,235 | USD      121,264 |

## Posicion Consolidada
| Moneda | Saldo total |
|---|---|
| PEN | S/ 580,939.20 |
| USD | USD 118,639.42 |

## Notas del Agente
- CUENTA-001: flujo neto positivo esta semana (PEN +1,969/dia). Revisar pagos proximos.
- CUENTA-002: flujo neto positivo esta semana (PEN +1,289/dia). Revisar pagos proximos.
- CUENTA-003: flujo neto positivo esta semana (USD +187/dia). Revisar pagos proximos.

Reporte guardado: D:\Proyecto_Gabriel\Skill_financiero\data\reporte_

---
## Verificacion final — los 3 SKILLs y sus artefactos

In [8]:
print('Estado de la Capa 1 (SKILLs) y Capa 2 (Scripts):')
print()
for skill_dir in sorted(SKILLS_BASE.iterdir()):
    if skill_dir.is_dir():
        archivos = list(skill_dir.iterdir())
        print(f'.claude/skills/{skill_dir.name}/')
        for a in sorted(archivos):
            size = a.stat().st_size
            print(f'  {a.name:<35} {size:>6} bytes')
print()

# Verificaciones
assert (SKILLS_BASE / 'forecast-cashflow' / 'SKILL.md').exists()
assert (SKILLS_BASE / 'forecast-cashflow' / 'rolling_mean_cashflow.py').exists()
assert (SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py').exists()
assert (SKILLS_BASE / 'alerta-tesoreria'  / 'SKILL.md').exists()
assert (SKILLS_BASE / 'reporte-semanal'   / 'SKILL.md').exists()
assert (SKILLS_BASE / 'reporte-semanal'   / 'generar_reporte.py').exists()
assert (BASE / 'data' / 'reporte_semanal_ultimo.md').exists()

print('OK: Todas las verificaciones pasaron')
print('Las Capas 1 y 2 del agente estan completas.')

Estado de la Capa 1 (SKILLs) y Capa 2 (Scripts):

.claude/skills/alerta-tesoreria/
  SKILL.md                              2400 bytes
.claude/skills/forecast-cashflow/
  batch_dias_de_caja.py                 2273 bytes
  rolling_mean_cashflow.py              2304 bytes
  SKILL.md                              2373 bytes
.claude/skills/reporte-semanal/
  generar_reporte.py                    5995 bytes
  SKILL.md                              2391 bytes

OK: Todas las verificaciones pasaron
Las Capas 1 y 2 del agente estan completas.


---
## Resumen NB 03

| Artefacto | Capa | Estado |
|---|---|---|
| `forecast-cashflow/SKILL.md` | 1 — Skills | OK (NB 02) |
| `forecast-cashflow/rolling_mean_cashflow.py` | 2 — Code Execution | OK (NB 02) |
| `forecast-cashflow/batch_dias_de_caja.py` | 2 — Code Execution | OK (NB 02) |
| `alerta-tesoreria/SKILL.md` | 1 — Skills | OK |
| `reporte-semanal/SKILL.md` | 1 — Skills | OK |
| `reporte-semanal/generar_reporte.py` | 2 — Code Execution | OK |
| `data/reporte_semanal_ultimo.md` | Output | OK |

**Las Capas 1 y 2 de la arquitectura del agente estan completas.**

**Siguiente:** `04_loop_agentico.ipynb`  
Conectaremos estos scripts al loop `while turns < max_turns` de la Messages API.  
El agente podra recibir preguntas en lenguaje natural y ejecutar los scripts correctos.